In [77]:
# from owner of this dataset

# About this Dataset
# Context
# This is a pre-crawled dataset, taken as subset of a bigger dataset (more than 4.7 million job listings) that was created by extracting data from Monster.com, a leading job board.

# Content
# This dataset has following fields:

# country
# country_code
# date_added
# has_expired - Always false.
# job_description - The primary field for this dataset, containing the bulk of the information on what the job is about.
# job_title
# job_type - The type of tasks and skills involved in the job. For example, "management".
# location
# organization
# page_url
# salary
# sector - The industry sector the job is in. For example, "Medical services".
# Acknowledgements
# This dataset was created by PromptCloud's in-house web-crawling service.

# Inspiration
# What kinds of jobs titles correspond with what kinds of wages?
# What can you learn about the Moster.com-based US job market based on analyzing the contents of the job descriptions?
# How do job descriptions different between different industry sectors?

In [78]:
# import tools and configs

import polars as pl
import pygwalker as pyg
import datetime

pl.Config(fmt_str_lengths=100, tbl_width_chars=200)

In [79]:
# Create Dataframe and having the first glace at the dataset.

df = pl.read_csv(r"C:\Users\Natth\Downloads\Neng_Data_Pipeline\dirty_data\monster\monster_com-job_sample.csv")



# df.schema

# df.filter(df['date_added'] != "").head()

In [80]:
# remove string in date_added.
# remove has_expired column because it only has "No".

df = df.drop(["has_expired", "date_added"])

# Lowering case everything.
df = df.with_columns(
    pl.col(pl.String).str.to_lowercase()
)


In [81]:
# because salary column is based on many formats Ex. hourly, yearly, even just text, so I will treat only numeric salary and change into yearly rates without bonuses (as they are the majority).

# Moving salary column to see it easier.
cols = [c for c in df.columns if c != "salary"]
cols.insert(3, "salary")
df = df.select(cols)

# removes spaces at start and end, $, and remaining space.
df = df.with_columns([
    pl.col("salary")
    .str.strip_chars()  
    .str.replace(r"^\$", "")
    .str.strip_chars()
    .alias("salary")
])

# Creating min and max salary columns.
df = df.with_columns([
        pl.col("salary").str.extract(r"([\d,]+\.?\d*)\s*-", 1).alias("started_from"),
        pl.col("salary").str.extract(r"-\s*([\d,]+\.?\d*)", 1).alias("up_to")
])

# remove "," in numbers to make them as float.
df = df.with_columns([
    pl.col("started_from").str.replace_all(",", "").cast(pl.Float64),
    pl.col("up_to").str.replace_all(",", "").cast(pl.Float64)
])

In [82]:

# Yearly salary converting
df_yearly = (
    df.filter(
        pl.col("salary").str.contains("year")
    ))

# some rows have small number assumedly to be hourly, I will take them out of df_yearly.
df_yearly = (df_yearly.filter(pl.col('up_to') > 1000))

#df_yearly.filter(pl.col("salary").is_not_null()).select(["salary","started_from", "up_to"]).unique()

In [83]:
# Monthly salary converting
df_monthly = (
    df.filter([
        pl.col("salary").str.contains("month")
    ]))

# some rows have small number assumedly to be hourly, and mistaken yearly salary. I will take them out of df_monthly.
df_monthly = (df_monthly.filter([pl.col('up_to') > 200, pl.col('started_from') <= 1800]))

#df_monthly.select("salary", "started_from", "up_to").unique().head(15)

In [84]:
# leftover salary converting (not yearly and monthly).

# combine yearly and monthly.
df_combined = pl.concat([df_yearly, df_monthly])

# subtract everything from df_combined, and remove all empty salary rows.
df_left = df.filter(
    ~pl.col("uniq_id").is_in(df_combined["uniq_id"])
)

df_left = df_left.with_columns(
    pl.col("salary").str.strip_chars().alias("salary")
).filter(
    pl.col("salary").is_not_null() & 
    (pl.col("salary") != "")
)

#df_left.select("salary").unique().show

C:\Users\Natth\AppData\Local\Temp\ipykernel_19736\834061017.py:7: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df_left = df.filter(


In [85]:

# Extract single number salary and put it in started_from.

df_left = df_left.with_columns(
    pl.col("salary").str.extract(r"([\d,]+\.?\d*)", 1).str.replace(",", "").alias("salary_single")
)

df_left = df_left.with_columns(
    pl.col("salary_single").cast(pl.Float64, strict=False).alias("salary_single")
)

df_left = df_left.with_columns(
    pl.col("started_from").fill_null(pl.col("salary_single"))
).drop("salary_single")


#df_left.show

In [86]:
# Hourly salary converting

df_hourly = df_left.filter(pl.col("started_from") < 150)

#df_hourly

In [87]:
# Convert hourly and monthly into yearly .

# Hourly.
df_hourly = df_hourly.with_columns([
    (pl.col("started_from") * 2080).alias("started_from"),
    (pl.col("up_to") * 2080).alias("up_to")
])

# Monthly.
df_monthly = df_monthly.with_columns([
    (pl.col("started_from") * 12).alias("started_from"),
    (pl.col("up_to") * 12).alias("up_to")
])

In [93]:
# Combine everything, remove unrealistic salaries (outliners), make average column, and round up the decimals.
df_final_salaries = pl.concat([df_yearly, df_monthly, df_hourly])

df_final_salaries = df_final_salaries.filter(
    (pl.col("up_to") > 4160) &
    (pl.col("up_to") <= 500000)
)

df_final_salaries = df_final_salaries.with_columns(
    pl.when(pl.col("up_to").is_null())
    .then(pl.col("up_to"))
    .otherwise((pl.col("up_to") + pl.col("up_to")) / 2)
    .alias("average_salary_per_year")
)

df_final_salaries = df_final_salaries.with_columns([
    pl.col("started_from").round(2),
    pl.col("up_to").round(2),
    pl.col("average_salary_per_year").round(2)
])

# Now we finished with salary.

In [ ]:
# Let's divide the sector and organization. Anything still unmatched goes to other, and if empty it will be not specified.

df = df_final_salaries.clone()

sector_map = [
    (r"it|software|database|desktop|systems analysis|web|ui|ux|software quality", "technology"),
    (r"account|finance|insurance|banking|real estate|mortgage|financial", "finance"),
    (r"medical|health|biotech|science|veterinary|animal", "healthcare"),
    (r"business|strategic|project|program|executive|svp|vp", "business & management"),
    (r"sales|marketing|business development|retail", "sales & marketing"),
    (r"human resources|administrative|clerical", "hr & admin"),
    (r"engineering|construction|trades|installation|maintenance|manufacturing|quality assurance", "engineering & trades"),
    (r"customer|support|food|hospitality", "customer & food services"),
    (r"legal|editorial|writing|security|logistics|transportation", "legal, education & other"),
    (r"manager|supervisor", "management"),
    (r"entry level|experienced|career level|student|education level", "other"),
]

df = df.with_columns(
    pl.when(pl.col("sector").is_null() | (pl.col("sector") == ""))
    .then(pl.lit("not specified"))
    .otherwise(pl.col("sector"))
    .alias("sector_group")
)

for pattern, group in sector_map:
    df = df.with_columns(
        pl.when(
            pl.col("sector").str.contains(pattern) | 
            pl.col("organization").str.contains(pattern)
        )
        .then(pl.lit(group))
        .otherwise(pl.col("sector_group"))
        .alias("sector_group")
    )

known_groups = [g for _, g in sector_map] + ["not specified"]
df = df.with_columns(
    pl.when(~pl.col("sector_group").is_in(known_groups))
    .then(pl.lit("other"))
    .otherwise(pl.col("sector_group"))
    .alias("sector_group")
)

#df.sort("sector_group", descending = False ).write_csv("test.csv")

In [105]:
# average salary per group.
df.group_by("sector_group").agg(
    pl.col("average_salary_per_year").mean().round(2).alias("avg_salary")
).sort("avg_salary", descending=True)

sector_group,avg_salary
str,f64
"""business & management""",105384.15
"""technology""",102173.86
"""healthcare""",85755.22
"""sales & marketing""",85649.82
"""management""",77415.56
…,…
"""engineering & trades""",64809.95
"""other""",62789.41
"""customer & food services""",53587.94
